# Related-Work-Adaption: gbert-large **Softmax** NER + HPO (ohne CRF)

Dieses Notebook ist die **Softmax-Baseline** zum CRF-Notebook
(`colab_train_gbert_crf.ipynb`) und dient als fairer Ablations-Partner: identischer
Datensatz (**GPTNERMED**), identisches BIO-Alignment, identische seqeval-Metriken und
dasselbe HPO-Budget. Der **einzige beabsichtigte Unterschied** ist der Kopf.

- **Kopf:** `gbert-large`-Encoder -> Dropout -> Linear -> **Softmax / Cross-Entropy**
  (Standard-`AutoModelForTokenClassification`). Jedes Token wird **unabhaengig**
  klassifiziert; es gibt keine gelernten Label-Uebergaenge wie beim CRF.
- **HPO-Buendel (Optuna):** TPE-Sampler + MedianPruner ueber Lernrate, Weight-Decay,
  Warmup, Dropout, Batch-Size und Epochen. (Kein `crf_lr_mult` — der Kopf ist nur ein
  Linear-Layer.)

> **Vergleich:** Stelle in *beiden* Notebooks `SEED`, `MAX_LENGTH` und das HPO-Budget
> gleich ein, dann sind Dev-/Test-F1 direkt vergleichbar und zeigen den echten Effekt
> des CRF-Kopfes.

> **Colab:** Runtime -> GPU (T4 reicht).


In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
REPO = "https://github.com/wieland-github/information_extraction_german_medical_data.git"
!rm -rf /content/information_extraction_german_medical_data
!git clone {REPO} /content/information_extraction_german_medical_data
%cd /content/information_extraction_german_medical_data

## Abhaengigkeiten

`datasets < 3.0` ist noetig, weil `jfrei/GPTNERMED` ueber ein Loading-Script geladen
wird (`trust_remote_code`). **Kein `pytorch-crf`** — dies ist die Softmax-Baseline.

In [ ]:
!pip install -q \
  "transformers>=4.40,<4.46" \
  "datasets>=2.16.0,<3.0.0" \
  "accelerate>=0.30" \
  "seqeval" \
  "optuna>=3.5"
import transformers, datasets, optuna, seqeval
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("optuna      ", optuna.__version__)

## Konfiguration

`TEST=True` fuehrt einen schnellen Smoke-Test aus (winziges Subset, 1 Epoche, 2 Trials).
Fuer einen fairen Vergleich: gleiche Werte wie im CRF-Notebook verwenden.

In [ ]:
import torch, os

MODEL_NAME  = "deepset/gbert-large"     # gleiches Backbone wie das Related-Work-Modell
DATASET     = "jfrei/GPTNERMED"
MAX_LENGTH  = 256
SEED        = 42

TEST = False   # True = Schnelltest (kleines Subset, 1 Epoche, 2 Trials)

# HPO-Budget (identisch zum CRF-Notebook halten)
N_TRIALS        = 2   if TEST else 8
HPO_TRAIN_LIMIT = 200 if TEST else 3000
HPO_EPOCHS      = 1   if TEST else 4      # feste Epochen je Trial; bestes Dev-F1 zaehlt

# Finales Training: grosszuegige Epochen-Obergrenze + Early Stopping legen die Laufzeit fest
FINAL_MAX_EPOCHS        = 2 if TEST else 15  # Obergrenze; Early Stopping bricht frueher ab
EARLY_STOPPING_PATIENCE = 1 if TEST else 3   # Epochen ohne Dev-F1-Verbesserung -> Stopp
FINAL_SEEDS             = [42] if TEST else [42]  # optional mehrere Seeds, z.B. [42, 52, 62]

DRIVE_DIR = "/content/drive/MyDrive/gbert_colab_softmax"
os.makedirs(DRIVE_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FP16   = torch.cuda.is_available()
print("Device:", DEVICE, "| fp16:", FP16, "| TEST:", TEST)

## Label-Schema (BIO)

Identisch zum CRF-Notebook. GPTNERMED liefert Zeichen-Spans mit Klassen-IDs
(`0=Medikation, 1=Dosis, 2=Diagnose`); daraus bauen wir ein BIO-Tagset mit `O`=ID 0.

In [ ]:
ID2CLASS = {0: "Medikation", 1: "Dosis", 2: "Diagnose"}

LABEL_LIST = ["O"]
for _cid in sorted(ID2CLASS):
    LABEL_LIST += [f"B-{ID2CLASS[_cid]}", f"I-{ID2CLASS[_cid]}"]

LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
NUM_LABELS = len(LABEL_LIST)
assert LABEL2ID["O"] == 0
print(NUM_LABELS, "Labels:", LABEL_LIST)

## Daten laden & Char-Spans -> BIO-Subword-Labels

Identisches Alignment wie im CRF-Notebook (wichtig fuer die Vergleichbarkeit):

- Spezial-Tokens (`[CLS]`, `[SEP]`) und Padding -> `-100` (aus Loss/Metrik ausgeschlossen).
- Erstes Subword einer Entitaet -> `B-…`, weitere Subwords -> `I-…`.
- Ein Token gehoert zu einer Entitaet, wenn es sich mit deren Zeichen-Span ueberlappt.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

raw = load_dataset(DATASET, trust_remote_code=True)
print(raw)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


def encode_example(example):
    text  = example["sentence"]
    ner   = example["ner_labels"]
    spans = list(zip(ner["start"], ner["stop"], ner["ner_class"]))

    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
    )

    labels = []
    prev_key = None
    for (tok_start, tok_end) in enc["offset_mapping"]:
        if tok_start == tok_end:            # Spezial-Token
            labels.append(-100)
            prev_key = None
            continue

        matched = None
        for (s, e, c) in spans:
            if tok_start < e and tok_end > s:
                matched = (s, e, c)
                break

        if matched is None:
            labels.append(LABEL2ID["O"])
            prev_key = None
        else:
            s, e, c = matched
            cls = ID2CLASS[int(c)]
            if prev_key == (s, e):
                labels.append(LABEL2ID[f"I-{cls}"])
            else:
                labels.append(LABEL2ID[f"B-{cls}"])
            prev_key = (s, e)

    enc.pop("offset_mapping")
    enc["labels"] = labels
    return enc


tokenized = raw.map(
    encode_example,
    remove_columns=raw["train"].column_names,
    desc="Tokenisieren + BIO-Alignment",
)
print(tokenized)

ex = tokenized["train"][0]
toks = tokenizer.convert_ids_to_tokens(ex["input_ids"])
print("\nBeispiel-Alignment:")
for t, l in list(zip(toks, ex["labels"]))[:25]:
    print(f"  {t:<20} {ID2LABEL.get(l, 'IGN') if l != -100 else 'IGN'}")

## Modell: gbert-Encoder + Linear + **Softmax** (Standard-Kopf)

`AutoModelForTokenClassification` haengt einen Dropout + Linear-Layer an den Encoder und
trainiert mit **tokenweiser Cross-Entropy** (Softmax ueber die Labels je Token). Die
`-100`-Positionen ignoriert der Loss automatisch. Es gibt **keine** Modellierung von
Label-Uebergaengen — genau das ist der Unterschied zum CRF.

Das tunebare `dropout` steuert `classifier_dropout` (Dropout direkt vor dem Kopf) und
entspricht damit dem Dropout im CRF-Modell.

In [ ]:
from transformers import AutoModelForTokenClassification


def build_model(dropout=0.1, model_name=None):
    return AutoModelForTokenClassification.from_pretrained(
        model_name or MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label={i: l for i, l in ID2LABEL.items()},
        label2id=LABEL2ID,
        classifier_dropout=dropout,
    )

## Metriken (seqeval)

`AutoModelForTokenClassification` liefert Logits `[B, T, C]`; die Vorhersage ist das
tokenweise **`argmax`** (bei reinem Softmax korrekt — es gibt keine Uebergaenge zu
dekodieren). `compute_metrics` mappt auf BIO-Strings und wertet mit seqeval aus.

In [ ]:
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds  = np.asarray(preds)
    labels = np.asarray(labels)
    if preds.ndim == 3:                     # Logits -> argmax
        preds = preds.argmax(-1)

    true_labels, true_preds = [], []
    for p_row, l_row in zip(preds, labels):
        cur_p, cur_l = [], []
        for p, l in zip(p_row, l_row):
            if l == -100:
                continue
            cur_l.append(ID2LABEL[int(l)])
            cur_p.append(ID2LABEL[int(p)] if int(p) >= 0 else "O")
        true_labels.append(cur_l)
        true_preds.append(cur_p)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall":    recall_score(true_labels, true_preds),
        "f1":        f1_score(true_labels, true_preds),
    }

## Data-Collator & TrainingArguments-Helfer

`DataCollatorForTokenClassification` paddet dynamisch und fuellt Label-Padding mit
`-100`. Es wird der **Standard-`Trainer`** genutzt (kein Custom-Trainer noetig — argmax
ist bei Softmax der korrekte Decoder). Mit `save_best=True` verhaelt sich das Training
wie spaCys `model-best`: nach jeder Epoche Dev-Eval, und am Ende wird der **beste
(Dev-F1)** Checkpoint zurueckgeladen (`load_best_model_at_end`) — Voraussetzung fuer
Early Stopping.

In [ ]:
from transformers import DataCollatorForTokenClassification, TrainingArguments, Trainer

collator = DataCollatorForTokenClassification(tokenizer=tokenizer, label_pad_token_id=-100)


def make_args(output_dir, lr, wd, warmup_ratio, epochs, batch_size, seed=SEED,
              save_best=False):
    common = dict(
        output_dir=output_dir,
        learning_rate=lr,
        weight_decay=wd,
        warmup_ratio=warmup_ratio,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=max(batch_size, 16),
        eval_strategy="epoch",
        logging_strategy="epoch",
        label_names=["labels"],
        remove_unused_columns=True,
        report_to="none",
        fp16=FP16,
        seed=seed,
        dataloader_num_workers=2,
        disable_tqdm=True,
    )
    if save_best:
        # bestes Modell auf Dev-F1 behalten (wie spaCys model-best) -> ermoeglicht Early Stopping
        common.update(
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="f1",    # sucht "eval_f1"
            greater_is_better=True,
            save_total_limit=1,
        )
    else:
        common.update(save_strategy="no")
    return TrainingArguments(**common)

# Synthtische Daten hinzufügen

In [ ]:
import json
from datasets import Dataset, concatenate_datasets

USE_SYNTHETIC  = True
# In Colab liegt die Datei auf Drive (wie in deinem meaningful_modification-Notebook):
SYNTHETIC_PATH = "/content/drive/MyDrive/synthetic_gptnermed.jsonl"

def load_synthetic_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

if USE_SYNTHETIC:
    syn_records = load_synthetic_jsonl(SYNTHETIC_PATH)
    print(f"Synthetische Saetze geladen: {len(syn_records)}")
    syn_ds  = Dataset.from_list(syn_records)
    syn_tok = syn_ds.map(encode_example, remove_columns=syn_ds.column_names,
                         desc="Synthetik: Tokenisieren + BIO-Alignment")
    # NUR ans Training anhaengen — validation/test bleiben Original-GPTNERMED!
    tokenized["train"] = concatenate_datasets([tokenized["train"], syn_tok])
    print("Train nach Merge:", len(tokenized["train"]))


## HPO-Buendel (Optuna)

**Suchraum** (identisch zum CRF-Notebook, ohne `crf_lr_mult`):

| Hyperparameter | Bereich | Begruendung |
|---|---|---|
| `learning_rate` | 1e-5 – 7e-5 (log) | Transformer-Feintuning-Fenster |
| `weight_decay` | 0.0 – 0.1 | Regularisierung |
| `warmup_ratio` | 0.0 – 0.2 | stabilerer Start |
| `dropout` | 0.1 – 0.4 | Regularisierung des Kopfes |
| `batch_size` | {8, 16} | GPU-Speicher vs. Gradientenrauschen |

Die **Epochenzahl wird nicht getunt**: jeder Trial laeuft feste `HPO_EPOCHS` Epochen und
als Score zaehlt das **beste Dev-F1 ueber die Epochen** (nicht die letzte) — konsistent
mit dem `model-best`-Prinzip des Finaltrainings.

TPE-Sampler + MedianPruner. Jeder Trial laeuft auf einem Train-Subset
(`HPO_TRAIN_LIMIT`); das finale Training nutzt den vollen Datensatz mit Early Stopping.

In [ ]:
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

hpo_train = tokenized["train"].shuffle(seed=SEED).select(
    range(min(HPO_TRAIN_LIMIT, len(tokenized["train"]))))
hpo_dev = tokenized["validation"]
print(f"HPO-Train: {len(hpo_train)}  |  HPO-Dev: {len(hpo_dev)}")


def objective(trial):
    lr         = trial.suggest_float("learning_rate", 1e-5, 7e-5, log=True)
    wd         = trial.suggest_float("weight_decay", 0.0, 0.1)
    warmup     = trial.suggest_float("warmup_ratio", 0.0, 0.2)
    dropout    = trial.suggest_float("dropout", 0.1, 0.4)
    batch_size = trial.suggest_categorical("batch_size", [8, 16])

    args = make_args(
        output_dir=f"/content/hpo/trial_{trial.number}",
        lr=lr, wd=wd, warmup_ratio=warmup, epochs=HPO_EPOCHS, batch_size=batch_size,
    )
    trainer = Trainer(
        model=build_model(dropout=dropout),
        args=args,
        train_dataset=hpo_train,
        eval_dataset=hpo_dev,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    # bestes Dev-F1 ueber alle Epochen (nicht nur die letzte) -> konsistent mit model-best
    dev_f1s = [h["eval_f1"] for h in trainer.state.log_history if "eval_f1" in h]
    best_f1 = max(dev_f1s) if dev_f1s else 0.0

    del trainer
    torch.cuda.empty_cache()
    return best_f1


study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_warmup_steps=1),
    study_name="gbert_softmax_ner",
)
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True)

print("\n==== HPO fertig ====")
print("Bester Dev-F1:", round(study.best_value, 4))
print("Beste Parameter:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

## Beste Parameter uebernehmen

Fallback-Werte, falls die HPO-Zelle uebersprungen wurde.

In [ ]:
try:
    best = dict(study.best_params)
except NameError:
    best = {}

BEST = {
    "learning_rate":    best.get("learning_rate", 3e-5),
    "weight_decay":     best.get("weight_decay", 0.01),
    "warmup_ratio":     best.get("warmup_ratio", 0.1),
    "dropout":          best.get("dropout", 0.2),
    "batch_size":       best.get("batch_size", 16),
}
# Die Epochenzahl kommt nicht aus der HPO -> Early Stopping auf Dev-F1 bestimmt sie.
print("Verwendete Hyperparameter fuers Finaltraining:")
BEST

## Finales Training: Early Stopping + Dev-bestes Modell

Wie spaCys `model-best`: Training mit grosszuegiger Obergrenze (`FINAL_MAX_EPOCHS`) auf
`train`, Evaluierung nach **jeder Epoche** auf `validation`, und via
`load_best_model_at_end` wird der **Checkpoint mit dem besten Dev-F1** zurueckgeladen.
**Early Stopping** (`EARLY_STOPPING_PATIENCE`) bricht ab, sobald sich das Dev-F1 nicht
mehr verbessert. Da das Modell ein `PreTrainedModel` ist, wird das beste Modell sauber
via `save_pretrained` gespeichert (Repo-`models`-Ordner **und** Google Drive).

In [ ]:
import json, shutil
from transformers import EarlyStoppingCallback

REPO_MODELS = "/content/information_extraction_german_medical_data/related_work/models"
best_overall = {"f1": -1.0, "dir": None, "seed": None}

for seed in FINAL_SEEDS:
    print(f"\n==================== FINAL seed {seed} ====================")
    args = make_args(
        output_dir=f"/content/final/seed_{seed}",
        lr=BEST["learning_rate"], wd=BEST["weight_decay"],
        warmup_ratio=BEST["warmup_ratio"], epochs=FINAL_MAX_EPOCHS,
        batch_size=BEST["batch_size"], seed=seed, save_best=True,
    )
    model = build_model(dropout=BEST["dropout"])
    trainer = Trainer(
        model=model, args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    trainer.train()
    # nach load_best_model_at_end ist trainer.model das dev-beste Modell
    dev_f1s = [h["eval_f1"] for h in trainer.state.log_history if "eval_f1" in h]
    print(f"Bestes Dev-F1 (seed {seed}): {max(dev_f1s):.4f}  (aus {len(dev_f1s)} Epochen)")

    test_metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
    print("Test:", {k: round(v, 4) for k, v in test_metrics.items()
                    if k.startswith("test_") and isinstance(v, float)})

    name = f"gbert-large-softmax-seed{seed}" + ("-test" if TEST else "")
    out_dir = os.path.join(REPO_MODELS, name)
    trainer.model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    with open(os.path.join(out_dir, "test_metrics.json"), "w") as f:
        json.dump(test_metrics, f, indent=2)
    with open(os.path.join(out_dir, "hyperparameters.json"), "w") as f:
        json.dump({"hyperparameters": BEST, "seed": seed}, f, indent=2)

    zip_path = shutil.make_archive(os.path.join(DRIVE_DIR, name), "zip", out_dir)
    print("Gespeichert in Drive:", zip_path)

    if test_metrics["test_f1"] > best_overall["f1"]:
        best_overall = {"f1": test_metrics["test_f1"], "dir": out_dir, "seed": seed}

    del trainer, model
    torch.cuda.empty_cache()

print("\nBestes Modell:", best_overall["dir"], "| Test-F1:", round(best_overall["f1"], 4))

## Detaillierter Report & Beispiel-Inferenz

Klassenweiser seqeval-Report des besten Modells und eine Beispiel-Vorhersage
(tokenweises argmax).

In [ ]:
# Bestes Modell laden
model = AutoModelForTokenClassification.from_pretrained(best_overall["dir"]).to(DEVICE).eval()

eval_args = make_args("/content/report", BEST["learning_rate"], 0.0, 0.0, 1,
                      BEST["batch_size"])
rep_trainer = Trainer(model=model, args=eval_args, data_collator=collator,
                      compute_metrics=compute_metrics)
pred = rep_trainer.predict(tokenized["test"])
logits = np.asarray(pred.predictions)
pred_ids = logits.argmax(-1)
y_pred, y_true = [], []
for p_row, l_row in zip(pred_ids, np.asarray(pred.label_ids)):
    cp, cl = [], []
    for p, l in zip(p_row, l_row):
        if l == -100:
            continue
        cl.append(ID2LABEL[int(l)])
        cp.append(ID2LABEL[int(p)])
    y_pred.append(cp); y_true.append(cl)
print(classification_report(y_true, y_pred, digits=4))


def predict(text):
    enc = tokenizer(text, return_offsets_mapping=True, truncation=True,
                    max_length=MAX_LENGTH, return_tensors="pt")
    offsets = enc.pop("offset_mapping")[0].tolist()
    with torch.no_grad():
        out = model(**{k: v.to(DEVICE) for k, v in enc.items()})
    tags = out.logits.argmax(-1)[0].tolist()
    ents, cur = [], None
    for (start, end), tag_id in zip(offsets, tags):
        if start == end:
            continue
        tag = ID2LABEL[tag_id]
        if tag.startswith("B-"):
            if cur: ents.append(cur)
            cur = {"label": tag[2:], "start": start, "end": end}
        elif tag.startswith("I-") and cur and cur["label"] == tag[2:]:
            cur["end"] = end
        else:
            if cur: ents.append(cur); cur = None
    if cur: ents.append(cur)
    return [(text[e["start"]:e["end"]], e["label"]) for e in ents]


print(predict("Der Patient erhielt 500 mg Ibuprofen bei Kopfschmerzen und Bluthochdruck."))